# FASE ANALÍTICA 2: La Desigualdad del Carbono (Auditado)
Este estudio acata la rigurosidad estadística definitiva: Deciles pesados por masa demográfica real, validación de GWP para Metano y curvas de Lorenz con Igualdad Perfecta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings = __import__('warnings')
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Purgado Sagrado (ISO Puro y NaNs Genuinos)
Aislamos únicamente naciones soberanas mediante `iso_code`. No se castigarán los datos faltantes rellenándolos con ceros; un origen sin medidas, se omite.

In [ ]:
url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
raw_df = pd.read_csv(url)

cols = ['country', 'iso_code', 'year', 'population', 'gdp', 'co2', 'co2_per_capita', 'methane', 'total_ghg']
df = raw_df[cols].copy()

# Filtrado ISO Puro: Eliminando falsos positivos (Mundo, Asia, High-Income)
df_countries = df[df['iso_code'].notna() & ~df['iso_code'].str.startswith('OWID', na=False)].copy()

# Riqueza Per Cápita estructurada
df_countries['gdp_per_capita'] = np.where((df_countries['population'].notna()) & (df_countries['population'] > 0), df_countries['gdp'] / df_countries['population'], np.nan)

print(f"Naciones Soberanas Viables Auditas: {df_countries['country'].nunique()}")
display(df_countries[['country', 'year', 'co2', 'methane']].tail(3))

## 2. El Crimen de la Física: Metano (CO2e) vs CO2 Puro
No se suman peras con manzanas. El metano calienta ~28 veces más que el CO2 en 100 años (GWP100). Verificamos la comparativa matemática del OWID (que suele reportarlo ya en CO2e, pero blindamos el cruce con `total_ghg`).

In [ ]:
df_ghg = df_countries.dropna(subset=['co2', 'methane']).copy()

# Agrupamos la evolución global desde 1990 (Protocolo de Kioto)
evol_ghg = df_ghg[df_ghg['year'] >= 1990].groupby('year')[['co2', 'methane', 'total_ghg']].sum()

plt.figure(figsize=(10, 5))
plt.plot(evol_ghg.index, evol_ghg['total_ghg'], label='Total GHG (Gases Efecto Invernadero)', color='black', linestyle='--', linewidth=2)
plt.plot(evol_ghg.index, evol_ghg['co2'], label='C02 Puro', color='#dc2626')
plt.plot(evol_ghg.index, evol_ghg['methane'], label='Metano (CO2 Equivalente)', color='#f59e0b')

plt.title('Evolución de Gases Equivalentes (GWP aplicable)', fontweight='bold')
plt.ylabel('Millones de Toneladas (CO2e)')
plt.legend()
plt.show()

## 3. La Curva de Lorenz: Desigualdad Demográfica Ponderada
Resolvemos *La Trampa de los Deciles*. Un decil de riqueza = 10% exacto de la población global (aprox 800M personas), y no un grupo del 10% de los países (independiente de su tamaño).

In [ ]:
df_2022 = df_countries[df_countries['year'] == 2022].copy()
df_lorenz = df_2022.dropna(subset=['gdp_per_capita', 'co2', 'population']).sort_values('gdp_per_capita')

# Calculamos el porcentaje acumulado de la Población Global
df_lorenz['pop_cum'] = df_lorenz['population'].cumsum()
df_lorenz['pop_share'] = df_lorenz['pop_cum'] / df_lorenz['population'].sum()

# Calculamos el porcentaje acumulado de la Emisión de C02
df_lorenz['co2_cum'] = df_lorenz['co2'].cumsum()
df_lorenz['co2_share'] = df_lorenz['co2_cum'] / df_lorenz['co2'].sum()

# Añadimos punto origen (0,0) para la curva pura
lorenz_pop = np.insert(df_lorenz['pop_share'].values, 0, 0)
lorenz_co2 = np.insert(df_lorenz['co2_share'].values, 0, 0)

plt.figure(figsize=(8, 8))
# Curva de Lorenz (Riqueza Poblacional vs Emisiones)
plt.plot(lorenz_pop, lorenz_co2, color='#ea580c', linewidth=3, label='Curva de Gini del Carbono')
# Línea Cero: La Perfecta Igualdad
plt.plot([0, 1], [0, 1], color='#10b981', linewidth=2, linestyle='--', label='Igualdad Perfecta (45°)')

plt.fill_between(lorenz_pop, lorenz_pop, lorenz_co2, color='#ea580c', alpha=0.1, label='Área de Desigualdad')

plt.title('Curva de Lorenz: Desigualdad del Carbono (2022)', fontweight='bold', fontsize=14)
plt.xlabel('Población Acumulada (% del Mundo, ordenados de Menor a Mayor Pobreza)')
plt.ylabel('Emisiones de C02 Acumuladas (% del Total Global)')
plt.legend(loc='upper left')
plt.show()

## 4. COPYWRITING AUTOMATIZADO (Extracción de Insights para React)
Calculando los extractos de la historia con cimientos estadísticos intachables.

In [ ]:
# Extraer 50% más Pobre
poorest_50_co2 = df_lorenz[df_lorenz['pop_share'] <= 0.50]['co2'].sum()
total_c02 = df_lorenz['co2'].sum()
porcentaje_pobre_50 = (poorest_50_co2 / total_c02) * 100

# Extraer 10% más Rico
richest_10_pop_threshold = 0.90
richest_10_co2 = df_lorenz[df_lorenz['pop_share'] >= richest_10_pop_threshold]['co2'].sum()
porcentaje_rico_10 = (richest_10_co2 / total_c02) * 100

# Extracción Histórica (USA vs China Cruce)
us_china = df_countries[df_countries['country'].isin(['United States', 'China'])]
pivot_cross = us_china.pivot(index='year', columns='country', values='co2').dropna()
cross_year = pivot_cross[pivot_cross['China'] > pivot_cross['United States']].index.min()

print("-----------------------------------------------------------------------------------------")
print("INSIGHTS NARRATIVOS CALCULADOS MATEMÁTICAMENTE (LISTOS PARA COPIAR EN REACT):\n")

# INSIGHT MEJORADO PARA MATIZ SENIOR (Intra-País Inequality Limit)
print(f"INSIGHT 1 (Desigualdad Estructural): 'Al agrupar a las naciones por su nivel de riqueza, el 10% más rico de la población global es responsable del {porcentaje_rico_10:.1f}% de las emisiones totales del planeta, mientras que la mitad inferior del mundo sobrevive emitiendo apenas un {porcentaje_pobre_50:.1f}%. Y esto es una estimación conservadora que no contabiliza la extrema desigualdad interna entre multimillonarios y ciudadanos de a pie.'\n")

print(f"INSIGHT 2 (El Relevo): 'Estados Unidos mantuvo la hegemonía durante un siglo, pero el dragón lo asfixió recientemente. Fue exactamente al cierre del año {int(cross_year)} cuando las industrias de China sobrepasaron estructuralmente el nivel bruto norteamericano.'\n")

us_ind = df_countries[(df_countries['country'].isin(['United States', 'India'])) & (df_countries['year'] == 2022)].set_index('country')
print(f"INSIGHT 3 (La Ilusión Bruta vs Per Cápita): 'India quema carbono salvajemente, sí. Pero la huella de responsabilidad individual cuenta otra historia. Mientras que un ciudadano indio promedio produce {us_ind.loc['India', 'co2_per_capita']:.1f} Toneladas por año, un ciudadano estadounidense dispara {us_ind.loc['United States', 'co2_per_capita']:.1f} Toneladas.'")
print("-----------------------------------------------------------------------------------------")